# Build GMAL JSON

Convert GMAL shapefile zips to compact JSON used by `gmal_3d_map_fixed (2).html`.

**Output per year:** `{ stats: {count,min,max,avg}, data: [[lon,lat,gmal,level,bus,rail,metro,llink], ...] }`

Coordinates reprojected from EPSG:27700 (British National Grid) to EPSG:4326 (WGS84).

- Input: `data_raw/`
- Output: ` ` 

## 1. Imports & paths

In [ ]:
import json
from pathlib import Path
import geopandas as gpd

HERE = Path.cwd()
RAW_DIR = HERE / "data_raw"
OUT_DIR = HERE 
OUT_DIR.mkdir(exist_ok=True)

YEARS = {
    "2016": RAW_DIR / "GMAL_grid_open_2016.zip",
    "2026": RAW_DIR / "GMAL_grid_open_2026.zip",
}
YEARS

{'2016': WindowsPath('c:/Users/Lenovo/Desktop/UCL_Moodle/MT2/CASA0029-Urban_Data_Visualisation/Assignments/Group_Visualisation/data_processing/GMAL_shapefiles_to_JSON/data_raw/GMAL_grid_open_2016.zip'),
 '2026': WindowsPath('c:/Users/Lenovo/Desktop/UCL_Moodle/MT2/CASA0029-Urban_Data_Visualisation/Assignments/Group_Visualisation/data_processing/GMAL_shapefiles_to_JSON/data_raw/GMAL_grid_open_2026.zip')}

## 2. Inspect a single shapefile

Quick peek at the 2026 dataset to confirm columns and CRS before running the full pipeline.

In [2]:
sample = gpd.read_file(f"zip://{YEARS['2026']}")
print("CRS:", sample.crs)
print("Rows:", len(sample))
sample.head()

CRS: EPSG:27700
Rows: 129109


,GridID,BusScore,RailScore,MetroScore,LLinkScore,LocalLink,GMALScore,GMALLevel,Altrincham,Ashton,...,McrAirport,McrMarktSt,McrOxfrdRd,McrSpnflds,Oldham,Rochdale,Stockport,TraffrdCtr,Wigan,geometry
0,1,0.0,0.0,0.0,0.0,No,0.0,1,None,None,...,None,None,None,None,None,None,None,None,None,"POLYGON ((389300 381100, 389300 381200, 389400..."
1,2,0.0,0.0,0.0,0.0,No,0.0,1,None,None,...,None,None,None,None,None,None,None,None,None,"POLYGON ((389400 381100, 389400 381200, 389500..."
2,3,0.0,0.0,0.0,0.0,No,0.0,1,None,None,...,None,None,None,None,None,None,None,None,None,"POLYGON ((389100 381200, 389100 381300, 389200..."
3,4,0.0,0.0,0.0,0.0,No,0.0,1,None,None,...,None,None,None,None,None,None,None,None,None,"POLYGON ((389200 381200, 389200 381300, 389300..."
4,5,0.0,0.0,0.0,0.0,No,0.0,1,None,None,...,None,None,None,None,None,None,None,None,None,"POLYGON ((389300 381200, 389300 381300, 389400..."


In [3]:
sample[["GMALScore", "GMALLevel", "BusScore", "RailScore", "MetroScore", "LLinkScore"]].describe()

,GMALScore,GMALLevel,BusScore,RailScore,MetroScore,LLinkScore
count,129109.000000,129109.000000,129109.000000,129109.000000,129109.000000,129109.000000
mean,4.922368,3.072714,3.632278,0.584021,0.399410,0.306659
std,6.797400,1.617594,4.671756,1.889544,1.282785,0.820130
min,0.000000,1.000000,0.000000,0.000000,0.000000,0.000000
25%,0.902307,2.000000,0.813242,0.000000,0.000000,0.000000
50%,3.484631,3.000000,2.879641,0.000000,0.000000,0.000000
75%,6.468691,4.000000,4.854108,0.000000,0.000000,0.000000
max,142.568230,8.000000,96.916219,30.278755,19.849063,2.500000


## 3. Build function

In [4]:
def build(zip_path: Path):
    gdf = gpd.read_file(f"zip://{zip_path}").to_crs(4326)
    cents = gdf.geometry.representative_point()
    rows = []
    scores = []
    for lon, lat, gmal, level, bus, rail, metro, llink in zip(
        cents.x, cents.y,
        gdf["GMALScore"], gdf["GMALLevel"],
        gdf["BusScore"], gdf["RailScore"], gdf["MetroScore"], gdf["LLinkScore"],
    ):
        rows.append([
            round(float(lon), 6), round(float(lat), 6),
            round(float(gmal), 2), int(level),
            round(float(bus), 2), round(float(rail), 2), round(float(metro), 2),
            round(float(llink), 2),
        ])
        scores.append(float(gmal))
    stats = {
        "count": len(rows),
        "min": round(min(scores), 2),
        "max": round(max(scores), 2),
        "avg": round(sum(scores) / len(scores), 2),
    }
    return {"stats": stats, "data": rows}

## 4. Run for each year and write to `data_processed/`

In [5]:
results = {}
for year, zip_path in YEARS.items():
    print(f"Building {year} from {zip_path.name}...")
    payload = build(zip_path)
    out = OUT_DIR / f"gmal_{year}.json"
    with out.open("w") as f:
        json.dump(payload, f, separators=(",", ":"))
    size_mb = out.stat().st_size / 1024 / 1024
    print(f"  -> {out.name}  {payload['stats']['count']:,} cells  {size_mb:.2f} MB")
    results[year] = payload

Building 2016 from GMAL_grid_open_2016.zip...
  -> gmal_2016.json  129,109 cells  5.63 MB
Building 2026 from GMAL_grid_open_2026.zip...
  -> gmal_2026.json  129,109 cells  5.62 MB


## 5. Inspect output stats

In [6]:
{year: payload["stats"] for year, payload in results.items()}

{'2016': {'count': 129109, 'min': 0.0, 'max': 158.65, 'avg': 6.14},
 '2026': {'count': 129109, 'min': 0.0, 'max': 142.57, 'avg': 4.92}}

In [7]:
results["2026"]["data"][:3]

[[-2.161362, 53.327158, 0.0, 1, 0.0, 0.0, 0.0, 0.0],
 [-2.159861, 53.32716, 0.0, 1, 0.0, 0.0, 0.0, 0.0],
 [-2.164369, 53.328053, 0.0, 1, 0.0, 0.0, 0.0, 0.0]]